In [1]:
import numpy as np
import numpy as np
from scipy.optimize import minimize
from qiskit.primitives import StatevectorEstimator
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import efficient_su2
import matplotlib.pyplot as plt
import time
from joblib import Parallel, delayed


In [2]:
# Define parameters

N = 6          # number of qubits
J = 1.0        # coupling strength
reps = 2
tol = 0.1 

# Define h values
h_values = np.concatenate([
    np.linspace(0.2, 0.8, 8),      # Ordered region 8 points
    np.linspace(0.8, 1.2, 10),     # Critical region 10 points 
    np.linspace(1.2, 3.0, 7)       # Disordered region 7 points
])    


# Finite-size scaling parameters (exact values for 1D TFIM)
N_values = [4, 6, 8, 10]    # system sizes to sweep
n_restarts = 1              # number of restarts per point
num_processes = 4           # number of processes for parallelization
beta_exp   = 0.125
nu_exp     = 1.0

In [ ]:
def build_tfim_hamiltonian(N: int, J: float, h: float) -> SparsePauliOp:
    pauli_terms = []
    for i in range(N - 1):
        pauli_str = ["I"] * N
        pauli_str[i]     = "Z"
        pauli_str[i + 1] = "Z"
        pauli_terms.append(("".join(reversed(pauli_str)), -J))
    for i in range(N):
        pauli_str = ["I"] * N
        pauli_str[i] = "X"
        pauli_terms.append(("".join(reversed(pauli_str)), -h))
    return SparsePauliOp.from_list(pauli_terms)


def build_ansatz(N: int, reps: int = 2):
    return efficient_su2(num_qubits=N, reps=reps, entanglement="linear")


def build_magnetisation_op(N: int) -> SparsePauliOp:
    pauli_terms = []
    for i in range(N):
        pauli_str = ["I"] * N
        pauli_str[i] = "Z"
        pauli_terms.append(("".join(reversed(pauli_str)), 1 / N))
    return SparsePauliOp.from_list(pauli_terms)


def exact_ground_state_energy(N: int, J: float, h: float) -> float:
    H_matrix = build_tfim_hamiltonian(N, J, h).to_matrix()
    return float(np.linalg.eigvalsh(H_matrix)[0])


def run_vqe(N: int, J: float, h: float, reps: int = 2,
            seed: int = 42, warm_start_params: np.ndarray = None):

    hamiltonian = build_tfim_hamiltonian(N, J, h)
    ansatz      = build_ansatz(N, reps)
    estimator   = StatevectorEstimator()

    def cost_fn(params):
        pub    = (ansatz, hamiltonian, params)
        result = estimator.run([pub]).result()
        return float(np.real(result[0].data.evs))

    # Determine if near critical point
    near_critical = abs(h / J - 1.0) < 0.3
    
    # Adaptive tolerances - Ordered/disordered regions converge faster with looser tolerances
    if near_critical:
        ftol = 1e-9  # Tight tolerance near critical point
        gtol = 1e-6
    else:
        ftol = 1e-6  # Looser tolerance away from critical point
        gtol = 1e-5
    
    # Adaptive maxiter
    if near_critical:
        maxiter = 500  # More iterations near critical point
    else:
        maxiter = 300  # Fewer iterations away from critical point

    attempts = n_restarts if (near_critical and warm_start_params is None) else 1
    

    # If warm start provided, use it as the sole starting point
    if warm_start_params is not None:
        starting_points = [warm_start_params]
    else:
        starting_points = [
            np.random.default_rng(seed + i).uniform(-np.pi, np.pi, ansatz.num_parameters)
            for i in range(attempts)
        ]

    best_energy = np.inf
    best_params = None

    for init_params in starting_points:
        result = minimize(
            cost_fn,
            init_params,
            method="L-BFGS-B",           # gradient-based, fast on statevector
            options={"maxiter": maxiter, "ftol": ftol, "gtol": gtol}
        )
        if result.fun < best_energy:
            best_energy = result.fun
            best_params = result.x

    return best_energy, best_params


def classify_phase(h: float, J: float, tol: float) -> str:
    ratio = h / J
    if abs(ratio - 1.0) < tol:
        return " CRITICAL "
    elif ratio < 1.0:
        return " ORDERED  "
    else:
        return "DISORDERED"


# Helper function for parallel processing
def process_single_h(h_index_tuple, N, J, h_values, reps, prev_params_dict, M_op, tol):
    """
    Process a single h value in parallel.
    Returns: (h_index, h, E_exact, E_vqe, M, params)
    """
    h_index, h = h_index_tuple
    phase = classify_phase(h, J, tol)
    print(f"[{h_index+1}/{len(h_values)}]  h/J = {h/J:.3f}  [{phase}]", end="  ")

    E_exact = exact_ground_state_energy(N, J, h)

    ansatz = build_ansatz(N, reps)

    # Use warm-start params if available from previous point
    prev_params = prev_params_dict.get(h_index - 1, None)
    E_vqe, params = run_vqe(N, J, h, reps=reps, warm_start_params=prev_params)

    # Calculate magnetisation
    estimator = StatevectorEstimator()
    pub    = (ansatz, M_op, params)
    result = estimator.run([pub]).result()
    M      = abs(float(np.real(result[0].data.evs)))

    print(f"E_exact = {E_exact:.4f}   E_vqe = {E_vqe:.4f}   "
          f"err = {abs(E_vqe - E_exact):.4f}   |M| = {M:.4f}")

    return (h_index, h, E_exact, E_vqe, M, params)


def sweep_phase_diagram(N: int, J: float, h_values: np.ndarray, reps: int = 2):
    """
    Returns (vqe_energies, exact_energies, magnetisations).
    Sequential sweep with warm starting.
    """
    vqe_energies   = []
    exact_energies = []
    magnetisations = []

    M_op         = build_magnetisation_op(N)
    estimator    = StatevectorEstimator()
    prev_params  = None

    for i, h in enumerate(h_values):
        phase = classify_phase(h, J, tol)
        print(f"[{i+1}/{len(h_values)}]  h/J = {h/J:.3f}  [{phase}]", end="  ")

        E_exact = exact_ground_state_energy(N, J, h)
        exact_energies.append(E_exact)

        ansatz        = build_ansatz(N, reps)
        E_vqe, params = run_vqe(N, J, h, reps=reps,
                                 warm_start_params=prev_params)
        prev_params   = params
        vqe_energies.append(E_vqe)

        pub    = (ansatz, M_op, params)
        result = estimator.run([pub]).result()
        M      = abs(float(np.real(result[0].data.evs)))
        magnetisations.append(M)

        print(f"E_exact = {E_exact:.4f}   E_vqe = {E_vqe:.4f}   "
              f"err = {abs(E_vqe - E_exact):.4f}   |M| = {M:.4f}")

    return (np.array(vqe_energies),
            np.array(exact_energies),
            np.array(magnetisations))


def run_single_N_sweep(N, J, h_values, reps):
    n_reps = reps if N <= 6 else reps + 1
    print(f"\n{'='*50}\nRunning sweep for N={N}, reps={n_reps}\n{'='*50}")
    start = time.time()
    vqe_energies, exact_energies, magnetisations = sweep_phase_diagram(
        N, J, h_values, reps=n_reps
    )
    print(f"Time for N={N}: {time.time() - start:.1f}s")
    return N, vqe_energies, exact_energies, magnetisations


def run_finite_size_sweep(N_values, J, h_values, reps=2):
    results = {}
    for N in N_values:
        n_reps = reps if N <= 6 else reps + 1
        print(f"\n{'='*50}")
        print(f"Running sweep for N={N}, reps={n_reps}")
        print(f"{'='*50}")
        start = time.time()

        vqe_energies, exact_energies, magnetisations = sweep_phase_diagram(
            N, J, h_values, reps=n_reps
        )

        elapsed = time.time() - start
        print(f"Time for N={N}: {elapsed:.1f}s")

        results[N] = {
            "vqe_energies"  : vqe_energies,
            "exact_energies": exact_energies,
            "magnetisations": magnetisations
        }
    return results

results = run_finite_size_sweep(N_values, J, h_values, reps=reps)


Running sweep for N=4, reps=2
[1/25]  h/J = 0.200  [ ORDERED  ]  E_exact = -3.0617   E_vqe = -3.0599   err = 0.0018   |M| = 0.9876
[2/25]  h/J = 0.286  [ ORDERED  ]  E_exact = -3.1294   E_vqe = -3.1220   err = 0.0074   |M| = 0.9746
[3/25]  h/J = 0.371  [ ORDERED  ]  E_exact = -3.2254   E_vqe = -3.2066   err = 0.0188   |M| = 0.9560
[4/25]  h/J = 0.457  [ ORDERED  ]  E_exact = -3.3520   E_vqe = -3.3125   err = 0.0395   |M| = 0.9327
[5/25]  h/J = 0.543  [ ORDERED  ]  E_exact = -3.5097   E_vqe = -3.4419   err = 0.0678   |M| = 0.9020
[6/25]  h/J = 0.629  [ ORDERED  ]  E_exact = -3.6969   E_vqe = -3.5930   err = 0.1039   |M| = 0.8648
[7/25]  h/J = 0.714  [ ORDERED  ]  E_exact = -3.9102   E_vqe = -3.8844   err = 0.0258   |M| = 0.0001
[8/25]  h/J = 0.800  [ ORDERED  ]  E_exact = -4.1456   E_vqe = -4.1240   err = 0.0216   |M| = 0.0000
[9/25]  h/J = 0.800  [ ORDERED  ]  E_exact = -4.1456   E_vqe = -4.1240   err = 0.0216   |M| = 0.0000
[10/25]  h/J = 0.844  [ ORDERED  ]  E_exact = -4.2749   E_vq

In [ ]:
# Plot the measurements

colors = ["steelblue", "teal", "coral", "purple"]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot Magnetisation vs h/J for increasing N
for (N, data), color in zip(results.items(), colors):
    axes[0].plot(h_values / J, data["magnetisations"],
                 "o-", label=f"N={N}", color=color)
axes[0].axvline(x=1.0, color="gray", linestyle=":", label="h/J = 1")
axes[0].set_xlabel("h / J")
axes[0].set_ylabel("|⟨M⟩|")
axes[0].set_title("Magnetisation vs h/J for increasing N")
axes[0].legend()


# Plot VQE error
for (N, data), color in zip(results.items(), colors):
    axes[1].plot(h_values / J,
                 np.abs(data["vqe_energies"] - data["exact_energies"]),
                 "o-", label=f"N={N}", color=color)
axes[1].axvline(x=1.0, color="gray", linestyle=":")
axes[1].set_xlabel("h / J")
axes[1].set_ylabel("|E_VQE - E_exact|")
axes[1].set_title("VQE error vs system size")
axes[1].legend()

plt.tight_layout()
plt.savefig("tfim_finite_size.png", dpi=150)
plt.show()

# Data collapse
fig, ax = plt.subplots(figsize=(8, 5))
for (N, data), color in zip(results.items(), colors):
    x_scaled = (h_values / J - 1.0) * (N ** (1.0 / nu_exp))
    y_scaled = data["magnetisations"] * (N ** (beta_exp / nu_exp))
    ax.plot(x_scaled, y_scaled, "o-", label=f"N={N}", color=color)
ax.set_xlabel(r"$(h/J - 1) \cdot N^{1/\nu}$")
ax.set_xlim(-8, 8) 
ax.set_ylabel(r"$|\langle M \rangle| \cdot N^{\beta/\nu}$")
ax.set_title("Finite-size scaling collapse (β=1/8, ν=1)")
ax.axvline(x=0, color="gray", linestyle=":", label="Critical point")
ax.legend()
plt.tight_layout()
plt.savefig("tfim_data_collapse.png", dpi=150)
plt.show()



for (N, data), color in zip(results.items(), colors):
    x_scaled = (h_values / J - 1.0) * (N ** (1.0 / nu_exp))

    # VQE magnetisation collapse
    y_vqe = data["magnetisations"] * (N ** (beta_exp / nu_exp))
    axes[0].plot(x_scaled, y_vqe, "o-", label=f"N={N}", color=color)

    # Exact magnetisation collapse — compute from exact ground state
    exact_mags = []
    for h in h_values:
        H_matrix  = build_tfim_hamiltonian(N, J, h).to_matrix()
        eigenvalues, eigenvectors = np.linalg.eigh(H_matrix)
        ground_state = eigenvectors[:, 0]

        M_op     = build_magnetisation_op(N)
        M_matrix = M_op.to_matrix()
        M_exact  = abs(float(np.real(ground_state.conj() @ M_matrix @ ground_state)))
        exact_mags.append(M_exact)

    y_exact = np.array(exact_mags) * (N ** (beta_exp / nu_exp))
    axes[1].plot(x_scaled, y_exact, "o-", label=f"N={N}", color=color)

for ax, title in zip(axes, ["VQE collapse", "Exact collapse"]):
    ax.axvline(x=0, color="gray", linestyle=":", label="Critical point")
    ax.set_xlabel(r"$(h/J - 1) \cdot N^{1/\nu}$")
    ax.set_ylabel(r"$|\langle M \rangle| \cdot N^{\beta/\nu}$")
    ax.set_title(f"Finite-size scaling collapse — {title} (β=1/8, ν=1)")
    ax.legend()
    ax.set_xlim(-8, 8)   # zoom in to the interesting region

plt.tight_layout()
plt.savefig("tfim_collapse_comparison.png", dpi=150)
plt.show()